In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install torch transformers datasets peft bitsandbytes py_vncorenlp \
    accelerate trl \
    numpy pandas scikit-learn sentencepiece tokenizers tqdm underthesea

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.4/978.4 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 43.1 MB/s eta 0:00:00
  Created wheel for py_vncorenlp: filename=py_vncorenlp-0.1.4-py3-none-any.whl size=4304 sha256=973457aa97ef54114bf3c83a40bc081cab56ebc1c43ae27b6d67c32638d3d7d8
  Stored in directory: /root/.cache/pip/wheels/db/e5/ff/f4a1b4ece36e8582db1ca71150a34e987e65df50c35974e9bb
Successfully built py_vncorenlp


In [3]:
!git clone https://github.com/Tommachilez/improving-learned-index.git
%cd /content/improving-learned-index

Cloning into 'improving-learned-index'...
remote: Enumerating objects: 1266, done.
remote: Counting objects: 100% (362/362), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 1266 (delta 314), reused 300 (delta 274), pack-reused 904 (from 1)
Receiving objects: 100% (1266/1266), 211.43 KiB | 8.46 MiB/s, done.
Resolving deltas: 100% (926/926), done.
/content/improving-learned-index


# Indexing

In [4]:
dataset_path = "/content/drive/MyDrive/KLTN_Project/Datasets"
doc_mapping = f"{dataset_path}/unique_doc_mapping.csv"
pretokenized_queries = f"{dataset_path}/vn_mining/queries_pretokenized.jsonl"
expanded_passages = f"{dataset_path}/passage_dir"
passage_tsv = f"{expanded_passages}/passages.tsv"
pid_mapping = f"{expanded_passages}/pid_mapping.txt"

model_checkpoint = "/content/drive/MyDrive/KLTN_Project/Models/deeperimpact_xlmr_vifc_maxp/DeepImpact_final.pt"
output_collection_path = f"{dataset_path}/deeperimpact_vifc_maxp/collection.index"
quantized_collection_path = f"{dataset_path}/deeperimpact_vifc_maxp/collection.index.quantized"
inverted_indices_path = f"{dataset_path}/deeperimpact_vifc_maxp/inverted_indices"
raw_run_file = f"{dataset_path}/deeperimpact_vifc_maxp/raw_run_file.txt"
run_file = f"{dataset_path}/deeperimpact_vifc_maxp/run_file.txt"

query_mapping = f"{dataset_path}/test_query_mapping.csv"
test_vifc = f"{dataset_path}/vifc_test_set.csv"
test_queries = f"{dataset_path}/test_queries.tsv"
test_qrels = f"{dataset_path}/test_qrels.tsv"

In [ ]:
!python -m src.deep_impact.scripts.create_passages \
    --input_csv {doc_mapping} \
    --queries_jsonl {pretokenized_queries} \
    --output_dir {expanded_passages}

Loading expansion terms from /content/drive/MyDrive/KLTN_Project/Datasets/vn_mining/queries_pretokenized.jsonl with max_terms=100...
Parsing queries: 61931it [00:01, 54145.30it/s]
Loaded expansion terms for 3030 documents.
Processing documents from /content/drive/MyDrive/KLTN_Project/Datasets/unique_doc_mapping.csv...
Creating passages: 3030it [00:00, 6566.16it/s]
Done. Processed 10027 total passages.
Outputs saved to:
  - Passages: /content/drive/MyDrive/KLTN_Project/Datasets/passage_dir/passages.tsv
  - Mapping:  /content/drive/MyDrive/KLTN_Project/Datasets/passage_dir/pid_mapping.txt


In [5]:
!python -m src.deep_impact.index \
  --collection_path {passage_tsv} \
  --output_file_path {output_collection_path} \
  --model_checkpoint_path {model_checkpoint} \
  --model_batch_size 32 \
#   --process_batch_size <process_batch_size> \
#   --num_processes <n> \

2026-01-02 12:57:08.377680: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767358628.399088    2127 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767358628.405010    2127 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767358628.419795    2127 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767358628.419821    2127 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767358628.419825    2127 computation_placer.cc:177] computation placer alr

In [6]:
!python -m src.deep_impact.indexing.quantize \
  -i {output_collection_path} \
  -o {quantized_collection_path}

10027it [00:01, 6502.25it/s]
Found max value: 1.6230000257492065
10027it [00:03, 2807.59it/s]


In [7]:
!python -m src.deep_impact.inverted_index.create \
  -i {quantized_collection_path} \
  -o {inverted_indices_path}

100% 10027/10027 [00:00<00:00, 11732.90it/s]
100% 50199/50199 [00:00<00:00, 1960627.87it/s]
100% 10027/10027 [00:01<00:00, 7186.46it/s]
50199it [00:01, 26450.80it/s]
100% 50199/50199 [00:00<00:00, 1283355.48it/s]


# Ranking

In [ ]:
!python -m src.deep_impact.scripts.create_test_files \
    --test_query_mapping {query_mapping} \
    --vifc_file {test_vifc} \
    --doc_mapping {doc_mapping} \
    --output_queries {test_queries} \
    --output_qrels {test_qrels}

Loading document mapping from /content/drive/MyDrive/KLTN_Project/Datasets/unique_doc_mapping.csv...
Loading Docs: 3030it [00:00, 26749.22it/s]
Loading VIFC data from /content/drive/MyDrive/KLTN_Project/Datasets/vifc_test_set.csv...
Loading VIFC: 7687it [00:00, 9788.00it/s] 
Processing queries and generating outputs...
Processing: 7685it [00:00, 21115.68it/s]

Processing Complete.
Generated 7685 queries in /content/drive/MyDrive/KLTN_Project/Datasets/test_queries.tsv
Generated 7686 qrels in /content/drive/MyDrive/KLTN_Project/Datasets/test_qrels.tsv


In [8]:
!python -m src.deep_impact.rank \
  --index_path {inverted_indices_path} \
  --queries_path {test_queries} \
  --qrels_path {test_qrels} \
  --output_path {raw_run_file} \
  --dataset_type msmarco \
#   --num_workers <n>

2026-01-02 13:04:20.783004: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767359060.802405    4072 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767359060.808282    4072 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767359060.823417    4072 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767359060.823444    4072 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767359060.823448    4072 computation_placer.cc:177] computation placer alr

In [9]:
!python -m src.deep_impact.aggregate_run \
    --run_file {raw_run_file} \
    --mapping {pid_mapping} \
    --output {run_file}

Loading ID mapping...
Aggregating results...
7685000it [00:10, 707525.01it/s]
Writing to /content/drive/MyDrive/KLTN_Project/Datasets/deeperimpact_vifc_maxp/run_file.txt...


In [10]:
!python -m src.deep_impact.evaluate \
  --run_file_path {run_file} \
  --qrels_path {test_qrels}

2026-01-02 13:24:08.158277: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767360248.189910    9112 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767360248.200405    9112 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767360248.224839    9112 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767360248.224875    9112 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767360248.224884    9112 computation_placer.cc:177] computation placer alr